<a href="https://colab.research.google.com/github/shwethashaji-ai/innothon-2026-india-2047-forecast/blob/main/Exxonmobil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup and Dataset Upload


In [1]:
import numpy as np, pandas as pd, io
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pd.set_option("display.float_format", "{:,.1f}".format)
df = pd.read_csv("renewable_energy_dataset.csv")      # file name must match what you uploaded
df.columns = [c.strip() for c in df.columns]
print(df.shape, df.Year.min(), "-", df.Year.max());
print(df.Zone.unique())

(635, 10) 2006 - 2024
['South' 'North-East' 'East' 'North' 'Central' 'West']


NPP generation history (TWh)


In [2]:
yrs = list(range(2007, 2025))   # FY ending year (2006-07 -> 2007)
hist = pd.DataFrame({
 "Thermal":[528490,560072,588282,640208,664822,708427,760454,792054,877941,943013,993516,1037152,1072311,1042763,1032514,1114726,1206211,1326545],
 "Hydro":[113502,120877,110098,104059,114415,130511,113720,134847,129244,121377,122378,126123,134894,155769,150300,151627,162099,134054],
 "Nuclear":[18802,16957,14927,18636,26266,32287,32866,34228,36102,37413,37916,38346,37813,46472,43029,47112,45861,47937],
 "RES":[9860,25210,27860,36947,39245,51226,47600,53100,51142,64339,78256,93429,97924,105074,111918,128781,172488,189490]},
 index=yrs) / 1000.0
hist["Total"] = hist.sum(axis=1);
hist.index.name = "FY"
print("Note: Thermal FY18, FY20, FY22, FY24 were read from the NPP chart line (about +/-25 GWh).")
hist.tail()

Note: Thermal FY18, FY20, FY22, FY24 were read from the NPP chart line (about +/-25 GWh).


,Thermal,Hydro,Nuclear,RES,Total
FY,,,,,
2020,"1,042.8",155.8,46.5,105.1,"1,350.1"
2021,"1,032.5",150.3,43.0,111.9,"1,337.8"
2022,"1,114.7",151.6,47.1,128.8,"1,442.2"
2023,"1,206.2",162.1,45.9,172.5,"1,586.7"
2024,"1,326.5",134.1,47.9,189.5,"1,698.0"


ML forecast to 2047

In [3]:
# ML: national generation forecast to 2047 (backtested model selection)
TARGET = 2047
def m_loglinear(y, h, win=10):
    ly = np.log(y[-win:]); x = np.arange(len(ly)); b = np.polyfit(x, ly, 1)
    return np.exp(b[1] + b[0]*(len(ly)-1+np.arange(1, h+1)))
def m_cagr(y, h, win=10, phi=0.97):
    g = (y[-1]/y[-win])**(1/(win-1)) - 1; v = y[-1]; out = []
    for _ in range(h): g *= phi; v *= 1+g; out.append(v)
    return np.array(out)
def m_holt(y, h):
    ly = np.log(y); best = None
    for a in np.linspace(.1,.9,9):
        for b in np.linspace(.05,.5,10):
            for ph in (.8,.9,.95):
                l, t, e = ly[0], ly[1]-ly[0], 0
                for v in ly[1:]:
                    p = l+ph*t; e += (v-p)**2; ln = a*v+(1-a)*p; t = b*(ln-l)+(1-b)*ph*t; l = ln
                if best is None or e < best[0]: best = (e, ph, l, t)
    _, ph, l, t = best
    return np.exp([l + sum(ph**k for k in range(1, i+1))*t for i in range(1, h+1)])
METHODS = {"loglinear": m_loglinear, "damped_cagr": m_cagr, "holt_damped": m_holt}

def backtest(y, horizon=3):
    err = {}
    for name, f in METHODS.items():
        es = []
        for hold in (3, 4, 5):
            tr, te = y[:-hold], y[-hold:][:horizon]
            es.append(np.mean(np.abs(f(tr, len(te)) - te) / te))
        err[name] = np.mean(es)
    return err

y = hist["Total"].values
err = backtest(y); best = min(err, key=err.get)
H = TARGET - hist.index[-1]
fc_years = list(range(hist.index[-1]+1, TARGET+1))
fc = METHODS[best](y, H)
mape = err[best]
# uncertainty band: backtest error widened with sqrt(horizon) (honest, not a statistical CI)
k = np.arange(1, H+1)
band = np.minimum(mape*np.sqrt(k/3), 0.35)
nat = pd.DataFrame({"Forecast_TWh": fc, "Low_TWh": fc*(1-band), "High_TWh": fc*(1+band)}, index=fc_years)
nat.index.name = "FY"
print("Backtest MAPE (3-yr ahead):", {k_: f"{v*100:.1f}%" for k_, v in err.items()})
print("Selected:", best)
print(nat.loc[[2030, 2035, 2040, 2047]].round(0))

Backtest MAPE (3-yr ahead): {'loglinear': '8.0%', 'damped_cagr': '6.8%', 'holt_damped': '7.7%'}
Selected: damped_cagr
      Forecast_TWh  Low_TWh  High_TWh
FY                                   
2030       2,211.0  1,998.0   2,425.0
2035       2,666.0  2,317.0   3,014.0
2040       3,131.0  2,637.0   3,625.0
2047       3,779.0  3,064.0   4,494.0


Regional population and GSDP

In [4]:
NPP_REGION = {
 "Northern":["Jammu And Kashmir","Ladakh","Himachal Pradesh","Punjab","Chandigarh","Haryana","Delhi","Uttarakhand","Uttar Pradesh","Rajasthan"],
 "Western":["Gujarat","Maharashtra","Goa","Madhya Pradesh","Chhattisgarh"],
 "Southern":["Andhra Pradesh","Telangana","Karnataka","Kerala","Tamil Nadu","Puducherry","Lakshadweep","Andaman And Nicobar Islands"],
 "Eastern":["Bihar","Jharkhand","West Bengal","Odisha","Sikkim"],
 "North Eastern":["Assam","Arunachal Pradesh","Nagaland","Manipur","Mizoram","Tripura","Meghalaya"]}
st2reg = {s:r for r,l in NPP_REGION.items() for s in l}
df["Region"] = df.State.map(st2reg)
print("Unmapped states:", sorted(df[df.Region.isna()].State.unique()))
df = df.dropna(subset=["Region"])
def damped_vals(s, n, win=10, phi=0.97):
    s = np.asarray(s, float); w = min(win, len(s)); g = (s[-1]/s[-w])**(1/(w-1)) - 1; v = s[-1]; out = []
    for _ in range(n): g *= phi; v *= 1+g; out.append(v)
    return np.array(out)
cnt = df.groupby("Year").State.nunique(); full = cnt[cnt == cnt.max()].index      # years where every state is present
z = df[df.Year.isin(full)].groupby(["Region", "Year"]).agg(Pop=("Population", "sum"), GSDP=("GSDP (crore INR)", "sum")).reset_index()
last = int(z.Year.max()); n = TARGET - last
def bt(s, hold=3):          # hide the last 3 years, predict them, report % error
    return np.mean(np.abs(damped_vals(s[:-hold], hold) - s[-hold:]) / s[-hold:])
rows = []
for reg, g in z.groupby("Region"):
    g = g.sort_values("Year")
    p, q = g.Pop.values, g.GSDP.values
    rows.append(dict(Region=reg, Pop_2024_M=p[-1]/1e6, Pop_2047_M=damped_vals(p, n)[-1]/1e6, Pop_backtest_err_pct=bt(p)*100,
                     GSDP_2024=q[-1], GSDP_2047=damped_vals(q, n)[-1], GSDP_backtest_err_pct=bt(q)*100))
R = pd.DataFrame(rows).set_index("Region")
R["Pop_growth_pct"] = (R.Pop_2047_M/R.Pop_2024_M - 1)*100
print("Years used:", full.min(), "-", full.max())
R.round(1).sort_values("Pop_2047_M", ascending=False)

Unmapped states: []
Years used: 2014 - 2024


,Pop_2024_M,Pop_2047_M,Pop_backtest_err_pct,GSDP_2024,GSDP_2047,GSDP_backtest_err_pct,Pop_growth_pct
Region,,,,,,,
Northern,437.4,534.3,0.5,"4,280,795.2","7,781,064.9",2.6,22.1
Western,319.4,385.7,0.3,"4,377,933.2","7,721,119.5",4.1,20.7
Eastern,315.4,376.7,0.2,"2,128,384.7","4,638,572.0",2.6,19.4
Southern,274.8,301.6,0.4,"4,936,747.5","10,643,623.4",4.7,9.7
North Eastern,51.9,61.5,0.2,"466,825.3","1,069,259.6",4.2,18.3


Driver-model test

In [5]:
# Driver model: generation per person ~ GSDP per person (population + economy), backtested vs the trend models
allyrs = df.Year.nunique()
keep = df.groupby("State").Year.nunique(); keep = keep[keep == allyrs].index      # states present in every year
nd = df[df.State.isin(keep)].groupby("Year").agg(Pop=("Population","sum"), GSDP=("GSDP (crore INR)","sum"))
nd = nd.loc[hist.index.min():hist.index.max()]
def damp(series, target, win=10, phi=0.97):
    s = series.values.astype(float); w = min(win, len(s)); g = (s[-1]/s[-w])**(1/(w-1)) - 1; v = s[-1]
    for _ in range(target - series.index[-1]): g *= phi; v *= 1+g
    return v
def path_drv(series, n, win=10, phi=0.97):
    s = series.values.astype(float); g = (s[-1]/s[-win])**(1/(win-1)) - 1; v = s[-1]; out = []
    for _ in range(n): g *= phi; v *= 1+g; out.append(v)
    return np.array(out)
def drv_fit_predict(gen, pop, gdp, n):
    x = np.log(gdp.values/pop.values); yv = np.log(gen.values/pop.values)
    b = np.polyfit(x, yv, 1)
    pf, gf = path_drv(pop, n), path_drv(gdp, n)
    return np.exp(b[1] + b[0]*np.log(gf/pf)) * pf, b[0]
g_tot = hist["Total"]
errs = []
for hold in (3, 4, 5):
    tr = slice(0, len(g_tot)-hold)
    p, _ = drv_fit_predict(g_tot.iloc[tr], nd.Pop.iloc[tr], nd.GSDP.iloc[tr], 3)
    te = g_tot.values[-hold:][:3]; errs.append(np.mean(np.abs(p[:len(te)]-te)/te))
err_drv = np.mean(errs)
print("Backtest MAPE: driver model %.1f%% vs best trend model (%s) %.1f%%" % (err_drv*100, best, mape*100))
f_drv, elast = drv_fit_predict(g_tot, nd.Pop, nd.GSDP, H)
print("Elasticity of generation/person to GSDP/person: %.2f" % elast)
print("Driver-model 2047: %.0f TWh | trend-model 2047: %.0f TWh" % (f_drv[-1], nat.loc[TARGET, "Forecast_TWh"]))
cmp_ = pd.DataFrame({"Trend_model": nat.Forecast_TWh, "Driver_model": f_drv}, index=nat.index); cmp_.loc[[2030, 2035, 2040, 2047]].round(0)

ok = (err_drv < 0.10) and (err_drv <= mape*1.25) and (0.5*nat.loc[TARGET,"Forecast_TWh"] < f_drv[-1] < 2*nat.loc[TARGET,"Forecast_TWh"])
print("Quality gate:", "PASS - driver model can be used" if ok else "FAIL - driver model rejected (worse backtest and implausible 2047 level); trend model stays the primary forecast")

Backtest MAPE: driver model 9.1% vs best trend model (damped_cagr) 6.8%
Elasticity of generation/person to GSDP/person: 0.37
Driver-model 2047: 1989 TWh | trend-model 2047: 3779 TWh
Quality gate: FAIL - driver model rejected (worse backtest and implausible 2047 level); trend model stays the primary forecast


cross-check against NPP year-to-date generation

In [6]:
ytd_fy27, ytd_fy26 = 1047369.79/1000, 959353.58/1000
g_ytd = ytd_fy27/ytd_fy26 - 1
g_mod = nat.loc[2027,"Forecast_TWh"]/nat.loc[2026,"Forecast_TWh"] - 1
print("NPP year-to-date growth FY27 vs FY26 : %.1f%%  |  model FY26->FY27 growth: %.1f%%" % (g_ytd*100, g_mod*100))
day27, day26 = 3977.55/1000, 3743.03/1000
print("Daily growth on 12 Nov (one day, noisy): %.1f%%" % ((day27/day26-1)*100))



NPP year-to-date growth FY27 vs FY26 : 9.2%  |  model FY26->FY27 growth: 4.6%
Daily growth on 12 Nov (one day, noisy): 6.3%


In [7]:
# 2047 source mix (NPP shares only) + criteria from NPP/dataset data only (no external sources)
src = ["Thermal", "Hydro", "Nuclear", "RES"]
S = hist[src].div(hist["Total"], axis=0)                       # shares of generation by year (NPP)
def alr(sh):
  return np.log(sh[["Hydro","Nuclear","RES"]].values / sh[["Thermal"]].values)       # log-ratio vs thermal
def inv(z):
    e = np.exp(np.c_[np.zeros(len(z)), z]); e = e/e.sum(axis=1, keepdims=True); return e[:, [0,1,2,3]]   # order: Thermal, Hydro, Nuclear, RES
def mix_path(sh, n, win=10, phi=0.95):
    z = alr(sh)[-win:]; x = np.arange(len(z));
    out = []
    sl = np.array([np.polyfit(x, z[:, j], 1)[0] for j in range(3)]);
    v = z[-1].copy()
    for _ in range(n):
      sl = sl*phi; v = v + sl;
      out.append(v.copy())      # damped trend of each log-ratio
    return inv(np.array(out))
# backtest: predict last 3 years of shares, compare with "no change" and report error in share points
errs, base = [], []
for hold in (3, 4, 5):
    tr = S.iloc[:-hold]; te = S.iloc[-hold:].values[:3]
    p = mix_path(tr, len(te));
    errs.append(np.abs(p - te).mean()*100);
    base.append(np.abs(tr.values[-1] - te).mean()*100)
print("Mix backtest (mean abs error, share points): model %.1f vs no-change %.1f" % (np.mean(errs), np.mean(base)))
mix = pd.DataFrame(mix_path(S, TARGET - S.index[-1]), index=range(S.index[-1]+1, TARGET+1), columns=src)
mix47 = mix.loc[TARGET];
twh47 = mix47*nat.loc[TARGET, "Forecast_TWh"]

def _dev(col, win=12):
    ly = np.log(hist[col].values[-win:]);
    x = np.arange(len(ly));
    b = np.polyfit(x, ly, 2);
    return np.std(ly - np.polyval(b, x))*100
vol = {c: _dev(c) for c in src}
card = pd.DataFrame({"FY2024": S.iloc[-1]*100, "FY2047": mix47*100}).round(1);
print(card)
res = {"Need_TWh_2047": nat.loc[TARGET, "Forecast_TWh"],
       "Fossil_(thermal)_share_pct_FY2024": S.iloc[-1]["Thermal"]*100, "Fossil_(thermal)_share_pct_2047": mix47["Thermal"]*100,
       "Concentration_HHI_FY2024": (S.iloc[-1]**2).sum(), "Concentration_HHI_2047": (mix47**2).sum(),
       "Availability_volatility_pct_2047": sum(mix47[k]*vol[k] for k in src)}
print(pd.Series(res).round(2))
print("Environment: judged only by the fossil (thermal) share of generation; no emission factors used (external).")
print("Cost: not calculated - no cost data in the dataset or NPP files.")
fig4 = go.Figure([go.Bar(name=k, x=["FY2024","FY2047"], y=[S.iloc[-1][k]*100, mix47[k]*100]) for k in src])
fig4.update_layout(barmode="stack", template="plotly_white", title="Generation mix: NPP FY2024 vs data-driven FY2047 (%)", height=420);
fig4.show()

Mix backtest (mean abs error, share points): model 0.9 vs no-change 1.1
         FY2024  FY2047
Thermal    78.1    61.5
Hydro       7.9     5.3
Nuclear     2.8     2.2
RES        11.2    31.0
Need_TWh_2047                       3,778.7
Fossil_(thermal)_share_pct_FY2024      78.1
Fossil_(thermal)_share_pct_2047        61.5
Concentration_HHI_FY2024                0.6
Concentration_HHI_2047                  0.5
Availability_volatility_pct_2047        5.3
dtype: float64
Environment: judged only by the fossil (thermal) share of generation; no emission factors used (external).
Cost: not calculated - no cost data in the dataset or NPP files.


Regional snapshot from the NPP dashboard (state-wise cumulative CONVENTIONAL generation)

In [8]:
D = {  # state: (region, 2026, 2025)
 "Uttar Pradesh":("Northern",116882.36,105467.19),"Rajasthan":("Northern",48261.69,41241.66),
 "Punjab":("Northern",24478.20,24078.05),"Haryana":("Northern",19600.85,16655.07),
 "Delhi":("Northern",2990.05,2013.44),"Uttarakhand":("Northern",12271.16,12922.26),
 "Himachal Pradesh":("Northern",32952.60,35928.42),
 "Jammu & Kashmir":("Northern",12424.96,13262.41),"Ladakh":("Northern",344.42,376.66),
 "Chhattisgarh":("Western",106434.84,101142.21),"Maharashtra":("Western",100285.23,91183.33),
 "Madhya Pradesh":("Western",90529.83,90913.69),"Gujarat":("Western",70191.80,57299.33),
 "Tamil Nadu":("Southern",63127.28,54246.95),"Andhra Pradesh":("Southern",56123.38,49093.71),
 "Telangana":("Southern",43572.34,39919.99),"Karnataka":("Southern",43005.55,37184.52),
 "Kerala":("Southern",3564.61,6812.29),
 "Odisha":("Eastern",48349.41,46350.53),"West Bengal":("Eastern",59469.81,57271.86),
 "Jharkhand":("Eastern",31404.76,23626.97),"Bihar":("Eastern",41671.04,35834.43),
 "Sikkim":("Eastern",3556.08,2233.56),
 "Assam":("North Eastern",6004.78,5662.12),"Tripura":("North Eastern",2310.13,2142.96),
 "Mizoram":("North Eastern",168.27,221.01),"Meghalaya":("North Eastern",662.16,778.80),
 "Nagaland":("North Eastern",188.68,195.91),"Arunachal Pradesh":("North Eastern",6976.48,3523.74),
}
snap = pd.DataFrame([(s,*v) for s,v in D.items()], columns=["State","Region","MU_2026","MU_2025"])
reg = snap.groupby("Region")[["MU_2026","MU_2025"]].sum()
reg["Growth_%"] = (reg.MU_2026/reg.MU_2025-1)*100
reg["Share_2026_%"] = reg.MU_2026/reg.MU_2026.sum()*100
reg["Share_2025_%"] = reg.MU_2025/reg.MU_2025.sum()*100
reg = reg.sort_values("MU_2026", ascending=False).round(1)
print(reg); print("\nTotal covered (MU):", round(reg.MU_2026.sum(),1), "vs", round(reg.MU_2025.sum(),1))
figs = go.Figure([go.Bar(name="2025", x=reg.index, y=reg["Share_2025_%"]), go.Bar(name="2026", x=reg.index, y=reg["Share_2026_%"])])
figs.update_layout(barmode="group", template="plotly_white", height=420, yaxis_title="% of covered total",
                   title="Regional share of conventional generation, 1 Apr-14 Nov (NPP dashboard)");
figs.show()


                MU_2026   MU_2025  Growth_%  Share_2026_%  Share_2025_%
Region                                                                 
Western       367,441.7 340,538.6       7.9          35.1          35.6
Northern      270,206.3 251,945.2       7.2          25.8          26.3
Southern      209,393.2 187,257.5      11.8          20.0          19.6
Eastern       184,451.1 165,317.4      11.6          17.6          17.3
North Eastern  16,310.5  12,524.5      30.2           1.6           1.3

Total covered (MU): 1047802.8 vs 957583.2


Dashboard

In [9]:
C = {"Thermal":"#4b5563","Hydro":"#2563eb","Nuclear":"#7c3aed","RES":"#16a34a"}
fig = make_subplots(rows=2, cols=2, specs=[[{"colspan":2}, None], [{"type":"xy"}, {"type":"xy"}]],
    subplot_titles=("National generation: NPP history and ML forecast to 2047 (TWh)",
                    "Regional population: 2024 vs 2047 forecast (million, dataset)", "Source mix of generation, NPP history (%)"),
    vertical_spacing=0.18)
for s in ["Thermal", "Hydro", "Nuclear", "RES"]:
    fig.add_trace(go.Scatter(x=hist.index, y=hist[s], name=s, stackgroup="h", line=dict(width=0.5, color=C[s])), 1, 1)
fig.add_trace(go.Scatter(x=nat.index, y=nat.High_TWh, line=dict(width=0), showlegend=False, hoverinfo="skip"), 1, 1)
fig.add_trace(go.Scatter(x=nat.index, y=nat.Low_TWh, fill="tonexty", fillcolor="rgba(17,24,39,0.12)", line=dict(width=0), name="Uncertainty range"), 1, 1)
fig.add_trace(go.Scatter(x=nat.index, y=nat.Forecast_TWh, name=f"ML forecast ({best})", line=dict(color="#111827", dash="dash", width=3)), 1, 1)
fig.add_annotation(x=TARGET, y=nat.Forecast_TWh[TARGET], text=f"2047: {nat.Forecast_TWh[TARGET]:,.0f} TWh", showarrow=True, ax=-60, ay=-40, row=1, col=1)
Rs = R.sort_values("Pop_2047_M", ascending=False)
fig.add_trace(go.Bar(x=Rs.index, y=Rs.Pop_2024_M, name="2024", marker_color="#94a3b8"), 2, 1)
fig.add_trace(go.Bar(x=Rs.index, y=Rs.Pop_2047_M, name="2047", marker_color="#0ea5e9"), 2, 1)
share = hist[["Thermal","Hydro","Nuclear","RES"]].div(hist["Total"], axis=0)*100
for s in ["Thermal", "Hydro", "Nuclear", "RES"]:
    fig.add_trace(go.Scatter(x=share.index, y=share[s], name=s, stackgroup="s", showlegend=False, line=dict(width=0.5, color=C[s])), 2, 2)
fig.update_layout(height=900, template="plotly_white", barmode="group",
    title="India electricity generation to 2047 (TWh generated basis). Regional generation: pending data",
    legend=dict(orientation="h", y=-0.08), margin=dict(t=90))
fig.show()
print("Source shares FY2024 (%):", share.iloc[-1].round(1).to_dict())

Source shares FY2024 (%): {'Thermal': 78.1, 'Hydro': 7.9, 'Nuclear': 2.8, 'RES': 11.2}


Regional 2047 generation

In [10]:
# Regional 2047 generation (TWh): national forecast x regional share. Dataset + NPP dashboard only; the weight is fitted, not assumed.
# Share_2047 = w*(today's NPP generation share) + (1-w)*(forecast share of population & GSDP, dataset). w is chosen by a backtest on the data.
cur = reg["Share_2026_%"]/100
drv = (R.Pop_2047_M/R.Pop_2047_M.sum() + R.GSDP_2047/R.GSDP_2047.sum())/2
cur, drv = cur.reindex(R.index), drv.reindex(R.index)
# --- choose w from data: predict the 2026 regional share from the 2025 share + latest dataset (2024) population/GSDP shares, compare with actual 2026 ---
ly = df[df.Year == df.Year.max()].groupby("Region").agg(P=("Population","sum"), G=("GSDP (crore INR)","sum"))
drv24 = ((ly.P/ly.P.sum() + ly.G/ly.G.sum())/2).reindex(R.index)
s25 = (reg["MU_2025"]/reg["MU_2025"].sum()).reindex(R.index); s26 = (reg["MU_2026"]/reg["MU_2026"].sum()).reindex(R.index)
def regional_(a, b, w):
    s = w*a + (1-w)*b; return s/s.sum()
grid = np.round(np.arange(0, 1.0001, 0.05), 2)
bt_err = pd.Series({w: (regional_(s25, drv24, w) - s26).abs().mean()*100 for w in grid})     # mean abs error, share points
W = float(bt_err.idxmin())
print("Backtest error by weight w (share points):\n", bt_err.round(2).rename(lambda w: f"{w:.2f}").to_string())
print(f"Weight chosen by the data: w = {W:.2f}  (error {bt_err.min():.2f} pts vs {bt_err[0]:.2f} pts for drivers-only and {bt_err[1.0]:.2f} pts for no-change)")
def regional(w): return regional_(cur, drv, w)
tab = pd.DataFrame(index=R.index)
tab["Share_now_%"] = cur*100; tab["Share_drivers_2047_%"] = drv*100
s = regional(W); tab["Share_2047_%"] = s*100
for c_, col in [("TWh_2047","Forecast_TWh"),("Low_TWh","Low_TWh"),("High_TWh","High_TWh")]: tab[c_] = s*nat.loc[TARGET, col]
tab["TWh_drivers_only"] = regional(0)*nat.loc[TARGET, "Forecast_TWh"]; tab["TWh_no_change"] = regional(1)*nat.loc[TARGET, "Forecast_TWh"]
tab = tab.sort_values("TWh_2047", ascending=False); print(tab.round(1).to_string())
figr = go.Figure(go.Bar(x=tab.index, y=tab.TWh_2047, error_y=dict(type="data", symmetric=False, array=tab.High_TWh-tab.TWh_2047, arrayminus=tab.TWh_2047-tab.Low_TWh),
                        text=[f"{v:,.0f}" for v in tab.TWh_2047], textposition="outside"))
figr.update_layout(template="plotly_white", height=450, yaxis_title="TWh generated, 2047", title="Regional generation 2047 (national forecast x data-fitted regional share)"); figr.show()


Backtest error by weight w (share points):
 0.00   4.0
0.05   3.8
0.10   3.6
0.15   3.4
0.20   3.2
0.25   3.0
0.30   2.8
0.35   2.6
0.40   2.4
0.45   2.1
0.50   1.9
0.55   1.8
0.60   1.6
0.65   1.3
0.70   1.1
0.75   0.9
0.80   0.7
0.85   0.5
0.90   0.4
0.95   0.3
1.00   0.4
Weight chosen by the data: w = 0.95  (error 0.26 pts vs 4.05 pts for drivers-only and 0.41 pts for no-change)
               Share_now_%  Share_drivers_2047_%  Share_2047_%  TWh_2047  Low_TWh  High_TWh  TWh_drivers_only  TWh_no_change
Region                                                                                                                      
Western               35.1                  23.7          34.5   1,303.6  1,057.0   1,550.2             897.0        1,325.0
Northern              25.8                  28.3          25.9     978.7    793.5   1,163.9           1,069.7          973.9
Southern              20.0                  25.8          20.3     765.9    621.0     910.9             974.7      

Tables and downloads

In [11]:
from google.colab import files
out = R.round(1)
display(out.sort_values("Pop_2047_M", ascending=False))
hist.to_csv("india_generation_history_TWh.csv"); nat.to_csv("national_forecast_2047_TWh.csv"); out.to_csv("regional_drivers_2047.csv")
reg.to_csv("regional_snapshot_conventional_ytd.csv")
for f in ["national_forecast_2047_TWh.csv","regional_drivers_2047.csv","india_generation_history_TWh.csv","regional_snapshot_conventional_ytd.csv"]: files.download(f)

,Pop_2024_M,Pop_2047_M,Pop_backtest_err_pct,GSDP_2024,GSDP_2047,GSDP_backtest_err_pct,Pop_growth_pct
Region,,,,,,,
Northern,437.4,534.3,0.5,"4,280,795.2","7,781,064.9",2.6,22.1
Western,319.4,385.7,0.3,"4,377,933.2","7,721,119.5",4.1,20.7
Eastern,315.4,376.7,0.2,"2,128,384.7","4,638,572.0",2.6,19.4
Southern,274.8,301.6,0.4,"4,936,747.5","10,643,623.4",4.7,9.7
North Eastern,51.9,61.5,0.2,"466,825.3","1,069,259.6",4.2,18.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# Damping sensitivity of the 2047 national forecast (same data, same model; shows how much the fixed damping setting matters)
def damped_total(y, h, win=10, phi=0.97):
    g = (y[-1]/y[-win])**(1/(win-1)) - 1; v = y[-1]
    for _ in range(h): g *= phi; v *= 1+g
    return v
def backtest_err(y, win, phi, horizon=3):
    es = []
    for hold in (3, 4, 5):
        tr, te = y[:-hold], y[-hold:][:horizon]
        pred = []; g = (tr[-1]/tr[-win])**(1/(win-1)) - 1; v = tr[-1]
        for _ in range(len(te)): g *= phi; v *= 1+g; pred.append(v)
        es.append(np.mean(np.abs(np.array(pred) - te)/te))
    return np.mean(es)*100
yy = hist["Total"].values; HH = TARGET - hist.index[-1]
sens = pd.DataFrame([(phi, backtest_err(yy, 10, phi), damped_total(yy, HH, 10, phi)) for phi in (0.90, 0.93, 0.95, 0.97, 1.00)],
                    columns=["Damping", "Backtest_error_%", "TWh_2047"])
shown = pd.DataFrame({"Damping": sens.Damping.map("{:.2f}".format), "Backtest_error_%": sens["Backtest_error_%"].map("{:.2f}".format), "TWh_2047": sens.TWh_2047.map("{:,.0f}".format)})
print(shown.to_string(index=False))
print("\nBacktest error changes by only %.2f points, while the 2047 total moves from %s to %s TWh." % (sens["Backtest_error_%"].max()-sens["Backtest_error_%"].min(), f"{sens.TWh_2047.min():,.0f}", f"{sens.TWh_2047.max():,.0f}"))
print("Damping 0.97 (used) gives %.0f TWh." % damped_total(yy, HH, 10, 0.97))

Damping Backtest_error_% TWh_2047
   0.90             6.71    2,546
   0.93             6.76    2,890
   0.95             6.80    3,246
   0.97             6.83    3,779
   1.00             6.89    5,217

Backtest error changes by only 0.18 points, while the 2047 total moves from 2,546 to 5,217 TWh.
Damping 0.97 (used) gives 3779 TWh.
